### Exercise 6: Intent-Based Routing (Middleware)


Topic: Middleware & RunnableBranch


Goal: Swap system prompts based on user input.

In [6]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [12]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableBranch, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-120b")
# Create a classification chain that classifies the user query into "tech" or "billing".
classification_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify the user query into one word: tech or billing."),
    ("human", "{input}"),
])

classification_chain = classification_prompt | llm | StrOutputParser()

# technical support chain
tech_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a Senior Developer. Use technical terms."),
    ("human", "{input}")
])

tech_chain = tech_prompt | llm | StrOutputParser()

# billing support chain
billing_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a polite Billing Agent. Help with invoices."), 
    ("human", "{input}")
])

billing_chain = billing_prompt | llm | StrOutputParser()

# Create a router that routes to the appropriate chain based on the classification.
router = RunnableBranch(
    (lambda x: 'tech' in x['topic'].lower(), tech_chain),
    (lambda x: 'billing' in x['topic'].lower(), billing_chain),
    billing_chain
)

# Create a full chain that first classifies the query and then routes to the appropriate chain.
full_chain = (
    RunnableLambda(lambda x: {"input": x, "topic": classification_chain.invoke({'input': x})}) | router
)

# Test the full chain with different queries.
print('Tech question:')
print(full_chain.invoke("How do I reset my api key"))

# Test the full chain with a billing question.
print('\nBilling question:')
print(full_chain.invoke("Where is my receipt?"))

Tech question:
Below is a concise, production‑grade workflow for rotating (resetting) an API key. The steps are deliberately generic so they apply to most SaaS platforms (OpenAI, AWS, GCP, Azure, etc.) while still giving you the concrete actions you need to take in the UI or via CLI.

---

## 1. Locate the Key Management UI / CLI Endpoint  

| Platform | UI Path | CLI / API |
|----------|---------|-----------|
| **OpenAI** | **Dashboard → API → Keys** | `openai api keys list` / `openai api keys create` |
| **AWS** | **IAM → Users → Security credentials** | `aws iam create-access-key` |
| **GCP** | **IAM & Admin → Service Accounts → Keys** | `gcloud iam service-accounts keys create` |
| **Azure** | **Portal → Azure AD → App registrations → Certificates & secrets** | `az ad app credential reset` |
| **Generic SaaS** | **Account → Security → API Tokens** | Check the provider’s REST endpoint for `POST /v1/api-keys` |

If you’re not sure which portal you’re on, look for a “Security”, “Acces